## Session 3.2: The Temporal Convolutional Network (TCN)

### The problem the TCN solves

Your encoder just produced `H` of shape `(batch, 512, L)` — a rich representation of the mixture at each time frame. Now the separator needs to answer: **at each time frame, which parts of this representation belong to speaker 1, and which to speaker 2?**

But here's the catch. You can't answer that question by looking at one frame in isolation. Whether a sound at frame 100 belongs to speaker 1 depends on what happened at frames 50, 60, 70 — and maybe even frames 200, 300 ahead. Speaker separation requires **long-range temporal context**.

How long? A typical utterance lasts 1-3 seconds. At your stride of 8 samples, that's roughly 2,000-24,000 frames of context you'd ideally want to see. That's the problem the TCN has to solve.

### Naive solution and why it fails

The obvious approach: use a large convolutional kernel. A kernel of size 10,000 would cover 5 seconds of context. What's wrong with this?

Two things. First, parameters — a single conv layer with kernel size 10,000 and 512 channels has `512 × 512 × 10,000 = 2.6 billion` parameters. Untrainable. Second, it still only captures *fixed-range* dependencies — everything within the kernel window equally, nothing outside.

### The TCN's actual solution: dilated convolutions

You covered dilation conceptually in Session 2.5. Now you need to understand it deeply enough to implement it.

A dilated convolution with dilation rate `d` inserts `d-1` gaps between each kernel element:

```
Normal conv (dilation=1), kernel size 3:
  Input:  [x0, x1, x2, x3, x4, x5, x6, x7]
  Kernel: [w0, w1, w2]
  Frame 0 sees: x0·w0 + x1·w1 + x2·w2   → receptive field: 3 samples

Dilated conv (dilation=2), kernel size 3:
  Frame 0 sees: x0·w0 + x2·w1 + x4·w2   → receptive field: 5 samples

Dilated conv (dilation=4), kernel size 3:
  Frame 0 sees: x0·w0 + x4·w1 + x8·w2   → receptive field: 9 samples
```

Same number of parameters (`3 × channels`), but the receptive field grows. Now stack these with **exponentially increasing dilation**: `1, 2, 4, 8, 16, 32, 64, 128` — that's 8 layers.

The receptive field after all 8 layers:

```
Layer 1 (d=1):   sees 3 frames
Layer 2 (d=2):   sees 5 more
Layer 3 (d=4):   sees 9 more
...
Layer 8 (d=128): sees 257 more

Total receptive field ≈ 2 × (1+2+4+8+16+32+64+128) = 510 frames
```

510 frames × stride 8 = 4,080 samples = ~0.25 seconds. Good but not enough. So the paper **repeats** this stack of 8 layers **3 times** (R=3 repetitions). Each repetition restarts dilation at 1 and goes to 128 again. Combined receptive field: ~1,530 frames ≈ 0.76 seconds of audio context. That's enough to capture the structure of most speech segments.

Here's what one TCN block looks like, and how they stack:

![Encoder Overview](assest/tcn_dilated_stack.svg)

### Inside one TCN block

Each TCN block is a residual block — a standard pattern you'll use constantly in deep learning. The key idea: the block adds its transformation to its input rather than replacing it. This lets gradients flow unchanged through the skip connection, enabling very deep networks to train.

One block does this:

```
input x: (batch, B, L)       B = bottleneck dimension = 128
  │
  ├─→ depthwise dilated Conv1d(kernel=3, dilation=d, padding=causal)
  │     → PReLU activation
  │     → LayerNorm
  │     → pointwise Conv1d (1×1, projects back to B channels)
  │     → output: (batch, B, L)
  │
  └─→ skip connection (residual): adds input directly to output
        → final output: (batch, B, L)
```

Two things to notice:

**Depthwise convolution:** Instead of applying all B filters to all B channels simultaneously (which is expensive), each filter only operates on *one* channel. This is `groups=B` in PyTorch's `nn.Conv1d`. Far fewer parameters.

**Causal padding:** To prevent the model from "seeing the future" during training, padding is only added to the *left* side of the sequence. You'll handle this by padding `(kernel_size-1) * dilation` zeros on the left and then trimming the output back to length `L`. This is important for eventual real-time use.

---

## TODO 1: Implement one TCN block

**Goal:** Build a `TCNBlock` class — a single residual dilated conv block.

**Requirements:**
- Takes `(batch, B, L)` as input, returns `(batch, B, L)` — same shape in and out
- Uses a depthwise `nn.Conv1d` with `groups=in_channels` for the dilated conv
- Uses `kernel_size=3` and configurable `dilation`
- Uses **causal padding** — pad left only, trim right after conv
- Uses `nn.PReLU` (not ReLU — the paper specifies PReLU, a learnable variant)
- Uses `nn.GroupNorm(1, channels)` for normalization (this is equivalent to LayerNorm for 1D sequences)
- Adds a residual (skip) connection — output = conv_path(x) + x
- Parameters to expose: `in_channels`, `dilation`

**Algorithm in plain English:**
```
1. depthwise conv: Conv1d(in_channels, in_channels, kernel=3, 
                          dilation=d, groups=in_channels, bias=False)
   → pad (kernel_size-1)*dilation zeros on the LEFT before this conv
   → trim (kernel_size-1)*dilation samples from the RIGHT after
   
2. PReLU activation

3. GroupNorm(1, in_channels)  ← normalizes across the channel dim

4. pointwise conv: Conv1d(in_channels, in_channels, kernel=1, bias=False)

5. residual: output = result_of_steps_1_4 + x
```

**Hints:**
- `F.pad(x, (left_pad, 0))` pads the left side of a tensor — look up `torch.nn.functional.pad`
- After padding and convolving, your sequence will be longer than `L` — you need `x[..., :L]` to trim it back
- `nn.PReLU()` has one learnable parameter per channel — check its docs for the `num_parameters` argument
- The residual addition only works if the conv path doesn't change the shape — verify your output is `(batch, B, L)` before adding

**Expected behavior:**
```python
block = TCNBlock(in_channels=128, dilation=1)
x = torch.randn(4, 128, 59999)
out = block(x)
assert out.shape == (4, 128, 59999)   # same shape in and out

block8 = TCNBlock(in_channels=128, dilation=128)
out8 = block8(x)
assert out8.shape == (4, 128, 59999)  # dilation doesn't change output shape
```

**After it works — experiment:**

Create 8 blocks with dilations `[1, 2, 4, 8, 16, 32, 64, 128]` and pass the same input through them in sequence. Confirm the shape is preserved all the way through. Then ask yourself: why does the output shape stay constant regardless of dilation? Where does the dilation actually *show up* in the computation?

Go implement — come back with code and your answer to that last question.

In [8]:
import torch
from torch import nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
import math 

from pathlib import Path
import os
import sys

# Check if __file__ exists (it won't in Jupyter)
try:
    current_dir = Path(__file__).parent
except NameError:
    # If in Jupyter, use the current working directory
    current_dir = Path(os.getcwd())

# Add project root to Python path
project_root = current_dir.parent.parent
sys.path.insert(0, str(project_root))

from src.data.loaders import get_dataloaders


# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

✅ Using Apple Silicon GPU
PyTorch version: 2.1.0
Device: mps


In [9]:
batch_size = 4

train_loader, val_loader, test_loader = get_dataloaders(
    train_manifest="../../data/processed/test/test_manifest.json",
    val_manifest="../../data/processed/train/train_manifest.json",
    test_manifest="../../data/processed/val/val_manifest.json",
    batch_size=batch_size,
)

# todo 1

In [10]:
class TCNBlock(nn.Module):
    def __init__(
            self, 
            in_channels=128, 
            kernal = 3,
            dilation=1
        ):
        super().__init__()
        
        self.kernal = kernal
        self.dilation = dilation
        
        self.conv = nn.Conv1d(
            in_channels=in_channels, 
            out_channels=in_channels, 
            kernel_size=3, 
            dilation=dilation, 
            groups=in_channels, 
            bias=False
            )
        self.prelu = nn.PReLU(in_channels)
        self.norm = nn.GroupNorm(1, in_channels)
        self.pointWise = nn.Conv1d(
            in_channels=in_channels, 
            out_channels=in_channels, 
            kernel_size=1, 
            bias=False
            )
        
    def forward(self, x):
        residual = x
        
        pad_L = (self.kernal - 1) * self.dilation
        
        out = F.pad(x, (pad_L, 0))
        out = self.conv(out)
        out = self.prelu(out)
        out = self.norm(out)
        out = self.pointWise(out)
        return out+residual

In [11]:
block = TCNBlock(in_channels=128, dilation=1)
x = torch.randn(4, 128, 59999)
out = block(x)
assert out.shape == (4, 128, 59999)   # same shape in and out

block8 = TCNBlock(in_channels=128, dilation=128)
out8 = block8(x)
assert out8.shape == (4, 128, 59999)  # dilation doesn't change output shape

In [12]:
dilations = [1, 2, 4, 8, 16, 32, 64, 128]

for dilation in dilations:
    block = TCNBlock(in_channels=128, dilation=dilation)
    x = torch.randn(4, 128, 59999)
    out = block(x)
    assert out.shape == (4, 128, 59999)   # same shape in and out

    block8 = TCNBlock(in_channels=128, dilation=dilation)
    out8 = block8(x)
    assert out8.shape == (4, 128, 59999)  # dilation doesn't change output shape
    print(f"dilation {dilation} run successfully ")

dilation 1 run successfully 
dilation 2 run successfully 
dilation 4 run successfully 
dilation 8 run successfully 
dilation 16 run successfully 
dilation 32 run successfully 
dilation 64 run successfully 
dilation 128 run successfully 
